# Rung 3 — two-species depletion

**What this validates:** the *multi-species* form of the White-Bear/BMCSL free energy.
Rungs 1 and 2 could both be passed by a single-species implementation; this one cannot.

**The setup.** Two hard-sphere species of different size. Only the **smaller** one is
coupled to the field ($\gamma_{\rm small} > 0$, $\gamma_{\rm big} = 0$). The big species
feels no field at all, so any structure in its profile can only come from volume
exclusion.

**Analytic answer.** With no field acting on it, the big species' equilibrium condition
is just

$$\ln \rho_{\rm big}(x) + \mu_{\rm ex}^{\rm big}(x) = \text{const},$$

where $\mu_{\rm ex}^{\rm big}$ is the insertion work for a big sphere into the *local
mixture* — a genuinely multi-species quantity, depending on both densities. So the
measured $-\ln \rho_{\rm big}(x)$ must reproduce the BMCSL insertion work evaluated at
the measured composition, up to one additive constant.

This is depletion: the big spheres are pushed out of the region the small spheres
crowd.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import (
    Series, project, mu_ex_from_profile, align_additive_constant,
    relative_discrepancy, report_comparison,
)

In [ ]:
SHAPE     = (32, 24)
VOXEL_NM  = 20.0
SIGMA_SMALL, SIGMA_BIG = 5.0, 8.0    # volume ratio ~4.1
CAP       = 20                        # chosen for the LARGEST species: 20 x 8nm -> eta 0.67
GAMMA_SMALL = 1.5
N_SMALL_PER_VOXEL, N_BIG_PER_VOXEL = 4, 2
N_STEPS   = 80_000
BURN_IN   = N_STEPS // 3
SAMPLE_EVERY = 25

TAU_S = suggest_tau(D_um2_s=1.0, voxel_nm=VOXEL_NM, dim=2, eta_max=0.45)
print(f"tau = {TAU_S:.3e} s")

ramp = np.arange(SHAPE[-1], dtype=float) / SHAPE[-1]
psi  = np.broadcast_to(ramp, SHAPE).copy()[None, ...]

sim = Simulation(
    shape=SHAPE, voxel_nm=VOXEL_NM,
    species=[
        Species("small", sigma_nm=SIGMA_SMALL, gamma=np.array([GAMMA_SMALL])),
        Species("big",   sigma_nm=SIGMA_BIG,   gamma=np.array([0.0])),   # no field
    ],
    occupancy_cap=CAP, psi=psi, D_um2_s=1.0, tau_s=TAU_S, seed=0,
)
sim.set_counts("small", np.full(SHAPE, N_SMALL_PER_VOXEL, dtype=np.int64))
sim.set_counts("big",   np.full(SHAPE, N_BIG_PER_VOXEL,   dtype=np.int64))
sim.record_initial()
sim

## Run

Both species start uniform. The small one develops a gradient because it feels the
field; the big one develops a gradient only because the small one is in its way.

In [ ]:
small, big = Series("small"), Series("big")
n_rows = SHAPE[0]

for i in range(N_STEPS):
    sim.step()
    if i >= BURN_IN and (i - BURN_IN) % SAMPLE_EVERY == 0:
        small.add(project(sim.state.lattice_view("small"), sim.lattice) / n_rows)
        big.add(project(sim.state.lattice_view("big"), sim.lattice) / n_rows)

sim.state.check_mass()
rho_s, rho_b = small.mean, big.mean
print(f"{small.n} samples")
print(f"small  {rho_s.min():.2f} - {rho_s.max():.2f} per voxel   (field-coupled)")
print(f"big    {rho_b.min():.2f} - {rho_b.max():.2f} per voxel   (gamma = 0)")
print(f"the big species varies by {(rho_b.max() / rho_b.min() - 1) * 100:.1f}% "
      f"across the box, from exclusion alone")

## Compare against the multi-species insertion work

The prediction is computed from the *measured* composition, so it is not an independent
curve — it is the statement that the profile is consistent with BMCSL. The composition
is a time-averaged mean and therefore fractional, which takes the elementwise
free-energy path rather than the integer table.

In [ ]:
composition = np.vstack([rho_s, rho_b])            # fractional: mean densities
exc = sim.exclusion

dnu_big = np.array([0, 1], dtype=np.int64)         # insert one big sphere
mu_big = (
    exc.shifted_free_energy(composition, exc.stoichiometry_offset(dnu_big), dnu_big)
    - exc.free_energy(composition)
)

measured = align_additive_constant(
    mu_ex_from_profile(rho_b, np.zeros_like(rho_b)), mu_big
)
sem = big.sem / rho_b

print(f"insertion work for a big sphere: {mu_big.min():.3f} - {mu_big.max():.3f} kT")
print()
print(report_comparison("-ln rho_big vs BMCSL big-sphere insertion work",
                        measured, mu_big, sem=sem))

VERDICTS = {"big-sphere insertion work": relative_discrepancy(measured, mu_big)["max"]}

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.5))

viz.plot_profile(rho_s, sem=small.sem, ax=ax[0], ylabel="mean occupancy",
                 label_measured="small (field-coupled)",
                 title="small species: driven by the field")
viz.plot_profile(rho_b, sem=big.sem, ax=ax[1], ylabel="mean occupancy",
                 label_measured=r"big ($\gamma=0$)",
                 title="big species: driven by depletion only")
viz.plot_profile(measured, predicted=mu_big, sem=sem, ax=ax[2],
                 ylabel=r"$\mu_{\rm ex}^{\rm big}$  ($k_BT$)",
                 label_measured=r"$-\ln\rho_{\rm big}$ + const",
                 label_predicted="BMCSL insertion work",
                 title="and they agree")
plt.tight_layout(); plt.show()

## What to take from this

The big species has **zero** coupling to the field, yet its density varies substantially
across the box. That structure is entropic: where small spheres are dense, there is less
room for a big one, so the big ones accumulate where the small ones are scarce. The
measured profile matches the BMCSL insertion work for a big sphere in the local mixture.

A single-species free energy cannot produce this. Rung 2 would pass with the
cross-species terms wrong; this one would not.

**Try changing:**

- `SIGMA_BIG = SIGMA_SMALL` — identical species, so no size asymmetry and no depletion.
  The big profile should flatten.
- `GAMMA_SMALL = 0` — nothing drives the small species, so nothing drives the big one
  either. Both go flat, and the comparison loses its dynamic range.
- `SIGMA_BIG = 12` — stronger asymmetry, stronger depletion. Note the cap: 20 × 12 nm
  spheres in a 20 nm voxel exceeds a packing fraction of 1, so construction will refuse
  it and tell you the largest admissible cap.